In [0]:
storage_account = "insurancestorage01"

spark.conf.set(
"fs.azure.account.key."+storage_account+".dfs.core.windows.net",
"azure_Storage_key"
)

spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

bronze_adls_path = "abfss://bronze@"+storage_account+".dfs.core.windows.net/hospitals/"
silver_adls_path = "abfss://silver@"+storage_account+".dfs.core.windows.net/hospitals/"

In [0]:
from pyspark.sql.types import *

hospital_schema = StructType([
    StructField("hospital_id", LongType(), True),
    StructField("hospital_name", StringType(), True),
    StructField("hospital_type", StringType(), True),
    StructField("network_flag", BooleanType(), True),
    StructField("empanelment_date", DateType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("region", StringType(), True),
    StructField("bed_capacity", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("average_claim_cost", DoubleType(), True)
])

In [0]:
hospitals_df = spark.read \
.option("header","true") \
.schema(hospital_schema) \
.csv(bronze_adls_path)

In [0]:
from pyspark.sql.functions import *

hospitals_df_silver = (
    hospitals_df

    # clean text columns
    .withColumn("hospital_name", trim(col("hospital_name")))
    .withColumn("hospital_type", trim(col("hospital_type")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
    .withColumn("region", trim(col("region")))

    # hospital tier classification
    .withColumn(
        "hospital_tier",
        when(col("bed_capacity") < 50, "Small")
        .when(col("bed_capacity") < 200, "Medium")
        .when(col("bed_capacity") < 500, "Large")
        .otherwise("Enterprise")
    )

    # high cost hospital indicator
    .withColumn(
        "high_cost_hospital_flag",
        when(col("average_claim_cost") > 300000, 1).otherwise(0)
    )

    # network category
    .withColumn(
        "network_category",
        when(col("network_flag") == True, "In-Network")
        .otherwise("Out-of-Network")
    )
)

In [0]:
from pyspark.sql.window import Window

window = Window.partitionBy("hospital_id").orderBy(col("empanelment_date").desc())

hospitals_df_silver = (
    hospitals_df_silver
    .withColumn("rnk", rank().over(window))
    .filter(col("rnk") == 1)
    .drop("rnk")
)

In [0]:
hospitals_df_silver_final = hospitals_df_silver.select(

    # identifiers
    "hospital_id",
    "hospital_name",

    # classification
    "hospital_type",
    "hospital_tier",

    # network info
    "network_flag",
    "network_category",
    "empanelment_date",

    # location
    "city",
    "state",
    "region",

    # capacity
    "bed_capacity",

    # performance
    "rating",
    "average_claim_cost",
    "high_cost_hospital_flag"
)

hospitals_df_silver_final.write \
.format("delta") \
.mode("overwrite") \
.save(silver_adls_path)